In [30]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df=pd.read_csv('/Categorical Data Encoding/data.csv')
df.head()

,movie_id,genre,country,language,age_rating,release_decade,subscription_plan,device,weekend_release,duration,imdb_score,production_budget_million,target_popular
0,1,Crime,UK,English,PG-13,1990s,Basic,Mobile,Yes,161,8.4,138.6,1
1,2,Documentary,South Korea,English,G,1980s,Basic,Mobile,Yes,146,5.6,164.2,0
2,3,Animation,South Korea,German,R,2020s,Standard,TV,Yes,164,6.7,73.1,0
3,4,Horror,Japan,Spanish,G,2010s,Basic,Tablet,No,152,5.9,15.6,0
4,5,Fantasy,Australia,Spanish,R,1980s,Premium,Tablet,No,148,5.5,22.0,0


In [31]:
df.dtypes

movie_id                       int64
genre                         object
country                       object
language                      object
age_rating                    object
release_decade                object
subscription_plan             object
device                        object
weekend_release               object
duration                       int64
imdb_score                   float64
production_budget_million    float64
target_popular                 int64
dtype: object

### Pandas

In [32]:
X = df.drop(columns="target_popular")
y = df["target_popular"]

X_train, X_test, y_train, y_test = train_test_split(X,y ,
                                   random_state=42,
                                   test_size=0.2,
                                   stratify=y)


In [33]:
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(X_train.shape)
print(X_test.shape)

(800, 54)
(200, 54)


In [34]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train,y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")

Accuracy: 0.915


### Scikit learn

In [35]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [36]:
cat_col = X_train.select_dtypes(include="object").columns

In [37]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            cat_col,
        )
    ],
    remainder="passthrough"
)

`handle_unknown="ignore"?`
If the test set contains a category that wasn't present during training, the encoder won't crash it will simply encode that category as all zeros. This is one of the main advantages over `pd.get_dummies()`.


In [38]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ))
    ]
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")

Accuracy: 0.915


## Conclusion

Both `pd.get_dummies()` and `OneHotEncoder()` achieved the same accuracy score (**0.915**) when used with a Random Forest classifier, indicating that the choice of encoding method did not affect the predictive performance on this dataset.

The main differences lie in their workflow and intended use. `pd.get_dummies()` is straightforward and well suited for exploratory analysis or small projects, but it requires manual alignment of columns between the training and test sets. In contrast, `OneHotEncoder()` integrates seamlessly with scikit-learn pipelines, automatically applies the same encoding to new data, and can safely handle unseen categories using `handle_unknown="ignore"`.

Overall, while both methods produced identical model performance in this experiment, `OneHotEncoder()` provides a more robust and scalable solution for real-world machine learning projects, whereas `pd.get_dummies()` offers a simpler approach for quick data preprocessing.



| Metric | `pd.get_dummies()` | `OneHotEncoder()` |
|---------|------------------|-----------------|
| Accuracy | **0.915** | **0.915** |
| Integration with scikit-learn Pipelines |  No |  Yes |
| Handles unseen categories |  Manual handling required |  `handle_unknown="ignore"` |
| Requires manual column alignment |  Yes (`reindex`) |  No |
| Returns | `pandas.DataFrame` | Sparse matrix (default) or dense array |
| Ease of implementation | Very simple | Slightly more complex |
| Best use case | Exploratory analysis and small projects | Machine learning pipelines and production workflows |